# Generation and ranking experiments

This notebook compares direct generation, candidate ranking and pooled candidate sources using
Qwen3-8B before and after the generator fine-tune.

The original experiment IDs are retained in `docs/arm_results.csv`:

| Arm | Method |
|---|---|
| 1 | Untuned generation with an instruction and six training examples |
| 1b | Untuned generation with the bare training prompt |
| 2 | Untuned ranking by generation likelihood |
| 3 | Fine-tuned generation |
| 4 | Fine-tuned generator used for likelihood ranking |
| 5 | Fine-tuned generator ranking pooled rule and generated candidates |

Notebook 05 adds the same-format untuned yes/no control and trained rankers with and without
mammalian GOSTAR. `WRITEUP.html` numbers its seven main configurations separately and includes
arm 2 among them. Arms 1b and 5 remain additional comparisons in the full results file.

Arms 1, 1b and 2 require inference calls but no new training. Arms 3, 4 and 5 use the checkpoint
recorded by notebook 03, while that checkpoint remains available.

## Setup

Scoring and the decoding policy come from modules, so every arm uses the same ones.

In [1]:
import datetime as dt
import json, os, pathlib, sys, time, warnings
from collections import defaultdict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

from src import evaluation as ev
from src import decoding as dec

# Credentials can be exported or read from the optional .env file.
from src.config import load_local_env, require_env
load_local_env(ROOT / ".env")

import tinker
import tinker.types as tt

BASE_MODEL = "Qwen/Qwen3-8B"
service = tinker.ServiceClient(api_key=require_env("TINKER_API_KEY"))
sampler = service.create_sampling_client(base_model=BASE_MODEL)
tokenizer = sampler.get_tokenizer()
print(f"connected, sampling from {BASE_MODEL}")

test = pd.read_csv(ROOT / "data" / "test_set.csv", dtype=str, keep_default_na=False, na_values=[])
test["parent"] = [ev.canonical(s) for s in test["parent_smiles"]]
test["metabolite"] = [ev.canonical(s) for s in test["metabolite_smiles"]]
parents = list(test["parent"])
references = list(test["metabolite"])
unique_parents = sorted(set(parents))
print(f"test set: {len(test)} pairs over {len(unique_parents)} molecules")

TODAY = dt.date.today().isoformat()

connected, sampling from Qwen/Qwen3-8B


test set: 309 pairs over 155 molecules


### The candidate lists for the ranking arms

The same 42 rules as `notebooks/01_enumeration_rules.ipynb`, applied once to each test molecule.

In [2]:
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem

RDLogger.DisableLog("rdApp.*")
rules = pd.read_csv(ROOT / "rules" / "metabolism_rules.csv")
COMPILED = []
for _, r in rules.iterrows():
    rxn = AllChem.ReactionFromSmarts(r["smirks"])
    rxn.Initialize()
    COMPILED.append((r["name"], rxn, bool(r["keep_largest_fragment"])))

def largest_fragment(mol):
    frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=False)
    return max(frags, key=lambda f: f.GetNumHeavyAtoms()) if len(frags) > 1 else mol

def enumerate_candidates(parent, per_rule_cap=200):
    mol = Chem.MolFromSmiles(parent)
    if mol is None:
        return []
    out = {}
    for name, rxn, keep in COMPILED:
        try:
            products = rxn.RunReactants((mol,))
        except Exception:
            continue
        n = 0
        for tup in products:
            for p in tup:
                try:
                    if keep:
                        p = largest_fragment(p)
                    Chem.SanitizeMol(p)
                    out.setdefault(Chem.MolToSmiles(p, canonical=True), name)
                except Exception:
                    pass
                n += 1
                if n > per_rule_cap:
                    break
            if n > per_rule_cap:
                break
    return list(out)

candidates = {p: enumerate_candidates(p) for p in unique_parents}
sizes = np.array([len(candidates[p]) for p in unique_parents])
covered = sum(
    1 for p, r in zip(parents, references)
    if ev.match_key(r, "standardized") in {ev.match_key(c, "standardized") for c in candidates[p]}
)
print(f"candidates per molecule: median {np.median(sizes):.0f}  mean {sizes.mean():.1f}  max {sizes.max()}")
print(f"the right answer is in the list for {covered} of {len(test)} pairs ({covered/len(test):.1%})")

candidates per molecule: median 32  mean 33.8  max 105
the right answer is in the list for 223 of 309 pairs (72.2%)


## Arm 1: Untuned, generative *(control)*

The base model writes metabolites from scratch, prompted with an instruction and six worked examples drawn from the **training** split.

In [3]:
from src.evaluation import canonical

train_examples = [json.loads(l) for l in (ROOT / "data" / "processed" / "train.jsonl").open()]
shots = []
seen_parents = set()
for row in train_examples:
    if row["is_canonical"] and row["parent"] not in seen_parents and 20 < len(row["parent"]) < 55:
        shots.append((row["parent"], row["target"]))
        seen_parents.add(row["parent"])
    if len(shots) == 6:
        break

INSTRUCTION = (
    "You are a drug metabolism expert. Given a parent drug molecule as SMILES, "
    "predict one metabolite the human body produces from it. "
    "Reply with the metabolite SMILES only, inside <metabolite></metabolite> tags.\n\n"
)
FEWSHOT = "".join(f"<parent>{p}</parent>\n<metabolite>{m}</metabolite>\n\n" for p, m in shots)

def build_prompt(parent):
    return INSTRUCTION + FEWSHOT + f"<parent>{parent}</parent>\n<metabolite>"

print(f"{len(shots)} worked examples in the prompt, {len(tokenizer.encode(build_prompt(unique_parents[0])))} tokens")
print()
print(build_prompt(unique_parents[0])[:380] + " ...")

6 worked examples in the prompt, 441 tokens

You are a drug metabolism expert. Given a parent drug molecule as SMILES, predict one metabolite the human body produces from it. Reply with the metabolite SMILES only, inside <metabolite></metabolite> tags.

<parent>CN(C)CC/C=C1/c2ccccc2COc2ccc(CC(=O)O)cc21</parent>
<metabolite>CNCCC=C1c2ccccc2COc2ccc(CC(=O)O)cc21</metabolite>

<parent>CCCCC/C=C\C/C=C\C/C=C\C/C=C\CCCC(=O)O</pa ...


### Spellings and temperatures

Generative arms sample several SMILES spellings and temperatures. The decoder combines votes for
equivalent structures. These sampling settings are shared across the generative arm comparison.

In [4]:
SPELLINGS = 4
SAMPLES_PER_CALL = 8
TEMPERATURES = (0.7, 1.0)
K = 5

from src import smiles as mu
import random

random.seed(0)
spellings = {p: (mu.get_smiles_variants(p, n=SPELLINGS) or [p]) for p in unique_parents}

n_calls = len(unique_parents) * SPELLINGS * len(TEMPERATURES)
prompt_tokens = np.mean([len(tokenizer.encode(build_prompt(p))) for p in unique_parents[:20]])
est_prefill = n_calls * prompt_tokens
est_output = n_calls * SAMPLES_PER_CALL * 24
print(f"  {n_calls:,} calls, {SAMPLES_PER_CALL} samples each = {n_calls*SAMPLES_PER_CALL:,} generations")
print(f"  estimated cost: prefill {est_prefill/1e6*0.195:.2f} USD + sampling {est_output/1e6*0.60:.2f} USD "
      f"= ~{est_prefill/1e6*0.195 + est_output/1e6*0.60:.2f} USD")

  1,240 calls, 8 samples each = 9,920 generations
  estimated cost: prefill 0.10 USD + sampling 0.14 USD = ~0.25 USD


### Smoke test on five molecules

Runs the whole path on five molecules: prompt the model, parse the replies, apply the policy, and confirm the predictions are valid SMILES.

An untuned model produces many broken SMILES on this task: unbalanced parentheses, impossible valences, and runaway repetition such as `CC1COC2COC3COC4COC5...` that only stops at the token limit. **That rate is part of arm 1's result.** The harness reports it as `invalid_rate`, and the fine-tuned arms are read against it.

The check below guards only against nothing being readable at all, which means the parser and the prompt disagree.

In [5]:

def sample_arm(client, prompt_fn, parent, spelling_list,
               temperatures=TEMPERATURES, n_samples=SAMPLES_PER_CALL, max_tokens=96):
    """Fire every (spelling, temperature) call for one molecule, then collect.

    The model and the prompt builder are arguments; everything else is fixed here. Arms differ
    in those two things and nothing else, so the sampling budget, the stop strings and the
    decoding that follows cannot drift between arms and be mistaken for a result.
    """
    futures = []
    for spelling in spelling_list:
        prompt = tt.ModelInput.from_ints(tokenizer.encode(prompt_fn(spelling)))
        for temp in temperatures:
            futures.append(client.sample(
                prompt=prompt, num_samples=n_samples,
                sampling_params=tt.SamplingParams(
                    max_tokens=max_tokens, temperature=temp, stop=["</metabolite>", "\n\n"])))
    return [[tokenizer.decode(s.tokens) for s in f.result().sequences] for f in futures]


def run_generating_arm(client, prompt_fn, label, report_every=40):
    """Run one generating arm over every test molecule, through the shared decoding policy."""
    t0 = time.time()
    decoded, raws, stats = {}, {}, []
    for i, p in enumerate(unique_parents, 1):
        raw = sample_arm(client, prompt_fn, p, spellings[p])
        raws[p] = raw
        d = dec.collapse(raw, p, k=K)
        decoded[p] = d
        stats.append({"parent": p, "raw": d.n_raw, "unparseable": d.n_unparseable,
                      "parent_echo": d.n_parent_echo, "distinct": d.n_distinct})
        if i % report_every == 0:
            print(f"  {label}: {i}/{len(unique_parents)}  ({time.time()-t0:.0f}s)")
    frame = pd.DataFrame(stats)
    print(f"  {label}: done in {time.time()-t0:.0f}s   "
          f"unreadable {frame['unparseable'].sum()/frame['raw'].sum():.1%}   "
          f"parent echoes {frame['parent_echo'].sum()/frame['raw'].sum():.1%}   "
          f"distinct offered (median) {frame['distinct'].median():.0f}")
    return decoded, raws, frame


# One scoring function for every ranking arm, for the same reason.
def score_candidates(client, parent, cands):
    """Mean log probability per completion token, under one model.

    Tokenised the way training tokenises: prefix and completion encoded separately and
    concatenated, exactly as ``to_datum`` builds a training example in notebook 03. Encoding the
    joint string instead lets BPE merge across the ``>`` boundary -- measured on 231 of 437 dev
    candidates (53%), with the completion length then miscounted for 180 (41%) -- so the
    normalising denominator would cover a different token set depending on the candidate's first
    character. Candidates whose first atom merges with the tag would be scored on a systematically
    more favourable basis than candidates whose first atom does not, within the same ranked list.
    """
    prefix_ids = tokenizer.encode(f"<parent>{parent}</parent>\n<metabolite>")
    futures = []
    for c in cands:
        completion_ids = tokenizer.encode(f"{c}</metabolite>")
        futures.append((len(completion_ids), client.compute_logprobs(
            prompt=tt.ModelInput.from_ints(prefix_ids + completion_ids))))
    scores = []
    for n_completion, fut in futures:
        tail = [x for x in fut.result()[-n_completion:] if x is not None]
        scores.append(float(np.mean(tail)) if tail else -1e9)
    return scores


def run_ranking_arm(client, candidate_map, label, report_every=40):
    """Rank a fixed candidate list per molecule by model likelihood."""
    t0 = time.time()
    out = {}
    for i, p in enumerate(unique_parents, 1):
        cands = candidate_map[p]
        if not cands:
            out[p] = dec.Decoded(predictions=[]); continue
        out[p] = dec.rank_candidates(cands, score_candidates(client, p, cands), p, k=K)
        if i % report_every == 0:
            print(f"  {label}: {i}/{len(unique_parents)}  ({time.time()-t0:.0f}s)")
    print(f"  {label}: done in {time.time()-t0:.0f}s")
    return out


# Smoke test on five molecules before spending on the full set.
t0 = time.time()
probe_raw = probe_unusable = 0
for p in unique_parents[:5]:
    raw = sample_arm(sampler, build_prompt, p, spellings[p])
    d = dec.collapse(raw, p, k=K)
    probe_raw += d.n_raw; probe_unusable += d.n_unparseable
    truth = {ev.match_key(r, "standardized") for q, r in zip(parents, references) if q == p}
    hit = [i for i, pred in enumerate(d.predictions, 1)
           if ev.match_key(pred, "standardized") in truth]
    print(f"  {p[:42]:<42} {d.n_distinct:>3} distinct from {d.n_raw:>3} samples, "
          f"{d.n_parent_echo:>2} echoes, {d.n_unparseable:>2} unusable"
          + (f"   HIT at rank {hit[0]}" if hit else ""))
print(f"\n  {time.time()-t0:.0f}s for 5 molecules")

# Low validity means a broken parser or a weak model; this threshold separates them.
# A parser bug once scored 98.7% of 9,920 good molecules as garbage.
rate = probe_unusable / max(probe_raw, 1)
print(f"  unusable: {probe_unusable}/{probe_raw} ({rate:.1%})")
assert rate < 0.9, (
    f"{rate:.1%} of replies could not be read at all, which looks like the parser and the prompt "
    f"disagreeing rather than a weak model. Check src/decoding.extract before spending on the "
    f"full set.")


  C#C[C@]1(O)CC[C@H]2[C@@H]3C(O)Cc4cc(O)ccc4  38 distinct from  64 samples,  0 echoes, 19 unusable


  C#C[C@]1(O)CC[C@H]2[C@@H]3CC(O)c4cc(O)ccc4  36 distinct from  64 samples,  3 echoes, 17 unusable


  C#C[C@]1(O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)CC  32 distinct from  64 samples,  0 echoes, 21 unusable


  C#C[C@]1(O)CC[C@H]2[C@@H]3CCC4=CC(=O)CC[C@  28 distinct from  64 samples,  5 echoes, 10 unusable


  C#C[C@]1(O)CC[C@H]2[C@@H]3CCc4c(ccc(O)c4O)  29 distinct from  64 samples,  1 echoes, 26 unusable

  25s for 5 molecules
  unusable: 93/320 (29.1%)


### The full run

In [6]:

arm1, arm1_raw, stats1 = run_generating_arm(sampler, build_prompt, "arm 1")


  arm 1: 40/155  (123s)


  arm 1: 80/155  (234s)


  arm 1: 120/155  (352s)


  arm 1: done in 433s   unreadable 24.3%   parent echoes 9.3%   distinct offered (median) 29


## Arm 2: Untuned ranking by likelihood

The base model scores the rule candidates by mean log probability per candidate token. This
controls the direct effect of summing over different output lengths; candidate length and
tokenization can still affect the scores. This arm appears in the main results table.


In [7]:

arm2 = run_ranking_arm(sampler, candidates, "arm 2")


  arm 2: 40/155  (91s)


  arm 2: 80/155  (158s)


  arm 2: 120/155  (218s)


  arm 2: done in 293s


## Arm 1b: Untuned generation with the bare prompt (additional control)

The base model receives the bare `<parent>...</parent>` prompt used for fine-tuning. Comparing
this arm with arm 1 measures the effect of replacing its instructions and examples with this
prompt.


In [8]:
def bare_prompt(parent):
    """The format the fine-tuned model is trained on, with no instruction and no examples."""
    return f"<parent>{parent}</parent>\n<metabolite>"

arm1b, arm1b_raw, stats1b = run_generating_arm(sampler, bare_prompt, "control")

  control: 40/155  (115s)


  control: 80/155  (219s)


  control: 120/155  (328s)


  control: done in 402s   unreadable 47.3%   parent echoes 10.5%   distinct offered (median) 22


In [9]:

def score_arm(decoded_by_parent, label):
    preds = [decoded_by_parent[p].predictions for p in parents]
    df = ev.evaluate(parents, references, preds, k_values=(1, 3, 5), criterion="standardized")
    df.insert(0, "arm", label)
    return df

baselines = pd.concat([score_arm(arm1,  "1  untuned, direct (few-shot)"),
                       score_arm(arm1b, "1b untuned, direct (bare)"),
                       score_arm(arm2,  "2  untuned, ranking")])

# Written out so no later notebook quotes a number by hand.
(ROOT / "docs" / "baselines.json").write_text(json.dumps({
    arm: {"recall@1": round(float(sub.loc[1, "recall_pair"]), 4),
          "recall@5": round(float(sub.loc[5, "recall_pair"]), 4),
          "precision@5": round(float(sub.loc[5, "precision_known"]), 4)}
    for arm, sub in baselines.groupby("arm")
}, indent=2))
baselines.round(3)


,arm,recall_pair,ceiling,pct_of_ceiling,recall_parent,parent_hit_rate,precision_known,tanimoto_wrong,parent_echo_rate
k,,,,,,,,,
1,"1 untuned, direct (few-shot)",0.016,0.515,0.031,0.018,0.032,0.032,0.427,0.0
3,"1 untuned, direct (few-shot)",0.036,0.864,0.041,0.035,0.065,0.022,0.522,0.0
5,"1 untuned, direct (few-shot)",0.049,0.964,0.050,0.055,0.084,0.018,0.534,0.0
1,"1b untuned, direct (bare)",0.000,0.515,0.000,0.000,0.000,0.000,0.148,0.0
3,"1b untuned, direct (bare)",0.006,0.864,0.007,0.013,0.013,0.004,0.290,0.0
5,"1b untuned, direct (bare)",0.013,0.964,0.013,0.026,0.026,0.005,0.339,0.0
1,"2 untuned, ranking",0.061,0.515,0.119,0.074,0.116,0.116,0.507,0.0
3,"2 untuned, ranking",0.146,0.864,0.169,0.157,0.258,0.095,0.558,0.0
5,"2 untuned, ranking",0.204,0.964,0.211,0.198,0.342,0.080,0.577,0.0


### Restricted to what the rules can reach

Arms 2 and 4 can only be right when the rules produced the right answer (72.2% of pairs), arm 3 has no such limit, and arm 5 has a different one. Scoring on the whole test set therefore mixes the rule set into the comparison. On the pairs where the answer is in the rule list, the arms answer the same question.

In [10]:

in_list = [
    ev.match_key(r, "standardized") in {ev.match_key(c, "standardized") for c in candidates[p]}
    for p, r in zip(parents, references)
]
sub_parents = [p for p, keep in zip(parents, in_list) if keep]
sub_refs = [r for r, keep in zip(references, in_list) if keep]
print(f"the rules can reach the answer for {len(sub_parents)} of {len(test)} pairs "
      f"({len(sub_parents)/len(test):.1%}); that subset is where the ranking arms are compared")


the rules can reach the answer for 223 of 309 pairs (72.2%); that subset is where the ranking arms are compared


## Arms 3, 4 and 5: the fine-tuned generative model

All three read one checkpoint, the best by dev recall, whose `tinker://` path comes from `data/processed/finetune_run.json`.

**Prompt asymmetry.** Arm 1 prompts the untuned model with an instruction and examples; arms 3 to 5 use the bare format the tuned model was trained on. Arm 1 against arm 3 therefore compares each model in the configuration that suits it, and arm 1b isolates the prompt.

In [11]:

RUN = json.loads((ROOT / "data" / "processed" / "finetune_run.json").read_text())
CHECKPOINT = RUN["best_checkpoint"]
tuned = service.create_sampling_client(model_path=CHECKPOINT)

print(f"  base model   : {RUN['base_model']}")
print(f"  best step    : {RUN['best_step']} of {RUN['training_steps']}")
print(f"  dev recall@5 : {RUN['full_dev_recall5']:.1%} on all {RUN['full_dev_molecules']} dev molecules")
print(f"  saved on     : {RUN['saved_on']}  (TTL {RUN['checkpoint_ttl_seconds']//86400} days)")
print(f"  checkpoint   : {CHECKPOINT}")

# A stale path fails deep inside the first arm with an opaque error, so check it here instead.
age_days = (dt.date.fromisoformat(TODAY) - dt.date.fromisoformat(RUN["saved_on"])).days
assert age_days * 86400 < RUN["checkpoint_ttl_seconds"], (
    f"the checkpoint was saved {age_days} days ago and its TTL is "
    f"{RUN['checkpoint_ttl_seconds']//86400} days, so it has expired. Re-run notebook 03.")

# Byte-for-byte the training format: a stray newline asks a question the model never saw.
TRAIN_PROMPT = json.loads((ROOT / "data" / "processed" / "train.jsonl").open().readline())["prompt"]
_parent_in_train = TRAIN_PROMPT.split("<parent>")[1].split("</parent>")[0]
assert bare_prompt(_parent_in_train) == TRAIN_PROMPT, \
    "bare_prompt() is not the format the model was trained on"
print(f"\n  prompt format verified against train.jsonl, checkpoint {age_days} day(s) old")


  base model   : Qwen/Qwen3-8B
  best step    : 1416 of 2130
  dev recall@5 : 29.3% on all 250 dev molecules
  saved on     : 2026-10-20  (TTL 7 days)
  checkpoint   : tinker://342f3b50-6665-5054-a274-23e8875898e4:train:0/sampler_weights/step1416



  prompt format verified against train.jsonl, checkpoint 0 day(s) old


### Arm 3: Fine-tuned, generative

The fine-tuned model writes answers, with the same decoding policy and k as arm 1.

In [12]:

arm3, arm3_raw, stats3 = run_generating_arm(tuned, bare_prompt, "arm 3")

print()
for label, frame in (("arm 1  untuned, few-shot", stats1),
                     ("control untuned, bare  ", stats1b),
                     ("arm 3  tuned,   bare   ", stats3)):
    print(f"  {label}  unreadable {frame['unparseable'].sum()/frame['raw'].sum():>6.1%}   "
          f"parent echoes {frame['parent_echo'].sum()/frame['raw'].sum():>6.1%}   "
          f"distinct offered (median) {frame['distinct'].median():>4.0f}")


  arm 3: 40/155  (331s)


  arm 3: 80/155  (813s)


  arm 3: 120/155  (1122s)


  arm 3: done in 1378s   unreadable 15.6%   parent echoes 6.1%   distinct offered (median) 32

  arm 1  untuned, few-shot  unreadable  24.3%   parent echoes   9.3%   distinct offered (median)   29
  control untuned, bare    unreadable  47.3%   parent echoes  10.5%   distinct offered (median)   22
  arm 3  tuned,   bare     unreadable  15.6%   parent echoes   6.1%   distinct offered (median)   32


### Arm 4: Fine-tuned generator used as ranker

The tuned generator scores the same rule candidates as arm 2 by mean candidate-token log
probability. The comparison changes which model supplies those likelihoods.


In [13]:

arm4 = run_ranking_arm(tuned, candidates, "arm 4")


  arm 4: 40/155  (274s)


  arm 4: 80/155  (543s)


  arm 4: 120/155  (812s)


  arm 4: done in 1050s


### Arm 5: Fine-tuned generator ranking pooled candidates

The tuned model ranks the union of the rule candidates and every distinct molecule arm 3
generated, including outputs outside its top five. This tests whether additional candidates
improve achieved recall as well as reachability.

In [14]:

POOL_K = 10**6     # collapse with no cap, to recover every distinct molecule arm 3 produced

def pooled_candidates(parent):
    """Rule candidates, plus every distinct molecule arm 3 generated that the rules missed."""
    generated = dec.collapse(arm3_raw[parent], parent, k=POOL_K).predictions
    rule_keys = {ev.match_key(c, "standardized") for c in candidates[parent]}
    return list(candidates[parent]) + [
        g for g in generated if ev.match_key(g, "standardized") not in rule_keys]

pool = {p: pooled_candidates(p) for p in unique_parents}
sizes = np.array([len(pool[p]) for p in unique_parents])
print(f"  pool size per molecule: median {np.median(sizes):.0f}  max {sizes.max()}  "
      f"(rules alone: median {np.median([len(candidates[p]) for p in unique_parents]):.0f})")

# If the model contributes no correct molecule the rules missed, arm 5 cannot beat arm 4.
def reachable(cmap):
    return sum(1 for p, r in zip(parents, references)
               if ev.match_key(r, "standardized") in
               {ev.match_key(c, "standardized") for c in cmap[p]})

r_rules, r_pool = reachable(candidates), reachable(pool)
print(f"  right answer reachable, rules only : {r_rules}/{len(test)} ({r_rules/len(test):.1%})")
print(f"  right answer reachable, pooled     : {r_pool}/{len(test)} ({r_pool/len(test):.1%})")
print(f"  the model contributed {r_pool - r_rules} answers the rules could not reach")

arm5 = run_ranking_arm(tuned, pool, "arm 5")


  pool size per molecule: median 62  max 153  (rules alone: median 32)
  right answer reachable, rules only : 223/309 (72.2%)
  right answer reachable, pooled     : 233/309 (75.4%)
  the model contributed 10 answers the rules could not reach


  arm 5: 40/155  (275s)


  arm 5: 80/155  (546s)


  arm 5: 120/155  (820s)


  arm 5: done in 1054s


## All six arms run here

In [15]:

ARMS = [("1  untuned, direct (few-shot)", arm1),
        ("1b untuned, direct (bare)",     arm1b),
        ("2  untuned, ranking",           arm2),
        ("3  tuned, direct",              arm3),
        ("4  tuned, ranking",             arm4),
        ("5  tuned, pooled",              arm5)]

results = pd.concat([score_arm(d, label) for label, d in ARMS])
results.to_csv(ROOT / "docs" / "arm_results.csv")
results.round(3)


,arm,recall_pair,ceiling,pct_of_ceiling,recall_parent,parent_hit_rate,precision_known,tanimoto_wrong,parent_echo_rate
k,,,,,,,,,
1,"1 untuned, direct (few-shot)",0.016,0.515,0.031,0.018,0.032,0.032,0.427,0.0
3,"1 untuned, direct (few-shot)",0.036,0.864,0.041,0.035,0.065,0.022,0.522,0.0
5,"1 untuned, direct (few-shot)",0.049,0.964,0.050,0.055,0.084,0.018,0.534,0.0
1,"1b untuned, direct (bare)",0.000,0.515,0.000,0.000,0.000,0.000,0.148,0.0
3,"1b untuned, direct (bare)",0.006,0.864,0.007,0.013,0.013,0.004,0.290,0.0
5,"1b untuned, direct (bare)",0.013,0.964,0.013,0.026,0.026,0.005,0.339,0.0
1,"2 untuned, ranking",0.061,0.515,0.119,0.074,0.116,0.116,0.507,0.0
3,"2 untuned, ranking",0.146,0.864,0.169,0.157,0.258,0.095,0.558,0.0
5,"2 untuned, ranking",0.204,0.964,0.211,0.198,0.342,0.080,0.577,0.0


### The comparison restricted to what the rules can reach

The same scoring, on the pairs whose answer is in the rule list.

In [16]:

restricted = pd.concat([
    ev.evaluate(sub_parents, sub_refs, [d[p].predictions for p in sub_parents],
                k_values=(1, 3, 5), criterion="standardized").assign(arm=label)
    for label, d in ARMS
])
restricted = restricted[["arm"] + [c for c in restricted.columns if c != "arm"]]
restricted.round(3)


,arm,recall_pair,ceiling,pct_of_ceiling,recall_parent,parent_hit_rate,precision_known,tanimoto_wrong,parent_echo_rate
k,,,,,,,,,
1,"1 untuned, direct (few-shot)",0.022,0.565,0.040,0.026,0.041,0.041,0.447,0.0
3,"1 untuned, direct (few-shot)",0.049,0.919,0.054,0.053,0.082,0.027,0.548,0.0
5,"1 untuned, direct (few-shot)",0.063,0.991,0.063,0.072,0.098,0.021,0.560,0.0
1,"1b untuned, direct (bare)",0.000,0.565,0.000,0.000,0.000,0.000,0.159,0.0
3,"1b untuned, direct (bare)",0.004,0.919,0.005,0.008,0.008,0.003,0.318,0.0
5,"1b untuned, direct (bare)",0.013,0.991,0.014,0.025,0.025,0.005,0.362,0.0
1,"2 untuned, ranking",0.085,0.565,0.151,0.104,0.148,0.148,0.539,0.0
3,"2 untuned, ranking",0.202,0.919,0.220,0.225,0.328,0.120,0.600,0.0
5,"2 untuned, ranking",0.283,0.991,0.285,0.292,0.434,0.102,0.625,0.0


In [ ]:
# The chart is drawn in 05_reranker.ipynb, which has arm 6.

at5 = pd.read_csv(ROOT / "docs" / "arm_results.csv", index_col=0).loc[5].set_index("arm")["recall_pair"]
pct = lambda a, b: (at5[b] - at5[a]) * 100

print("recall@5, and what each difference isolates:")
print(f"  instruction + examples, untuned : {at5[ARMS[1][0]]:>5.1%} -> {at5[ARMS[0][0]]:>5.1%}  "
      f"({pct(ARMS[1][0], ARMS[0][0]):+.1f}pp)")
print(f"  fine-tuning, prompt held fixed  : {at5[ARMS[1][0]]:>5.1%} -> {at5[ARMS[3][0]]:>5.1%}  "
      f"({pct(ARMS[1][0], ARMS[3][0]):+.1f}pp)")
print(f"  fine-tuning the ranker          : {at5[ARMS[2][0]]:>5.1%} -> {at5[ARMS[4][0]]:>5.1%}  "
      f"({pct(ARMS[2][0], ARMS[4][0]):+.1f}pp)")
print(f"  pooling both candidate sources  : {at5[ARMS[4][0]]:>5.1%} -> {at5[ARMS[5][0]]:>5.1%}  "
      f"({pct(ARMS[4][0], ARMS[5][0]):+.1f}pp)")

## The parent-echo rate

How often the model answers "nothing happens" by returning the parent unchanged. The decoder drops those before scoring, so `parent_echo_rate` in `docs/arm_results.csv` is 0 for every arm. The informative rate is the one measured before the filter, printed with each arm above.

| | parent echoes, before filtering |
|---|---:|
| untuned, instruction and examples | 9.3% |
| untuned, bare prompt | 10.5% |
| fine-tuned | 6.1% |

Roughly one reply in ten from an untuned model is the input handed back. Fine-tuning cuts that by a third, and the filter removes the rest.